# Intel x86 Assembly 명령어 처리: 단계별 분해 및 분류

본 노트북은 Intel x86/x64 어셈블리 명령어에서 다음을 수행합니다:
1. **Opcode와 Instruction 분리** - EVEX/VEX 접두사와 실제 명령어 분리  
2. **오퍼랜드 분리** - r/m16 같은 combined operand를 r16, m16으로 분할  
3. **오퍼랜드 그룹화** - 오퍼랜드를 의미별로 분류 (레지스터, 메모리, 즉시값 등)

## 단계별 진행

- **Sec 1**: instruction_list.csv 데이터 로드 및 분석
- **Sec 2**: Opcode와 Instruction 분리 로직 구현
- **Sec 3**: Combined Operand 파싱 및 분해
- **Sec 4**: 오퍼랜드 추출 및 중복 제거
- **Sec 5**: 오퍼랜드 의미별 분류  
- **Sec 6**: 최종 리포트 및 CSV 내보내기

## 섹션 1: 데이터 로드 및 현황 파악

현재 `instruction_list.csv`에 있는 명령어들을 로드하고, 다음을 확인합니다:
- 총 개수
- Opcode와 Instruction이 혼합되어 있는 패턴들
- Combined operand 패턴들 (r/m16, m16&16 등)

In [29]:
import pandas as pd
import re
from collections import defaultdict, Counter

# CSV 파일 로드
df = pd.read_csv('instruction_list.csv')
instructions = df['Instruction Syntax'].tolist()

print("=" * 80)
print("현재 instruction_list.csv 분석")
print("=" * 80)
print(f"\n✓ 총 명령어 수: {len(instructions)}")

# 패턴별 분류
opcode_mixed = []  # Opcode와 Instruction이 혼합된 것
combined_operands = []  # r/m16 같은 combined operand가 있는 것
simple_instructions = []  # 간단한 형태

for inst in instructions:
    if re.match(r'^(EVEX|VEX|REX)', inst):
        opcode_mixed.append(inst)
    elif '/' in inst or '&' in inst:
        combined_operands.append(inst)
    else:
        simple_instructions.append(inst)

print(f"\n📊 패턴 분류:")
print(f"  • Opcode-Instruction 혼합: {len(opcode_mixed):4}개 (예: EVEX.128... /rVMOVUPS)")
print(f"  • Combined Operand (/,&): {len(combined_operands):4}개 (예: r/m16)")
print(f"  • 간단한 명령어:           {len(simple_instructions):4}개 (예: ADD, MOV)")

# 샘플 표시
print(f"\n📌 [Opcode-Instruction 혼합 샘플]")
for i, item in enumerate(opcode_mixed[:5], 1):
    print(f"  {i}. {item[:70]}")

print(f"\n📌 [Combined Operand 샘플]")
for i, item in enumerate(combined_operands[:5], 1):
    print(f"  {i}. {item}")

현재 instruction_list.csv 분석

✓ 총 명령어 수: 5128

📊 패턴 분류:
  • Opcode-Instruction 혼합: 3463개 (예: EVEX.128... /rVMOVUPS)
  • Combined Operand (/,&):  603개 (예: r/m16)
  • 간단한 명령어:           1062개 (예: ADD, MOV)

📌 [Opcode-Instruction 혼합 샘플]
  1. EVEX-encoded 128-bit instructions
  2. EVEX-encoded 256-bit instructions
  3. EVEX-encoded 512-bit instructions
  4. EVEX.128.0F.W0 10 /rVMOVUPS xmm1 {k1}{z}
  5. EVEX.128.0F.W0 11 /rVMOVUPS xmm2/m128 {k1}{z}

📌 [Combined Operand 샘플]
  1. ADC r/m16
  2. ADC r/m32
  3. ADC r/m64
  4. ADC r/m81
  5. ADD r/m16


## 섹션 2: Opcode와 Instruction 분리

**목표**: EVEX/VEX/REX 접두사(Opcode)와 실제 명령어(Instruction)를 분리

**패턴 예시**:
- `EVEX.128.0F.W0 10 /rVMOVUPS xmm1 {k1}{z}`
  - Opcode: `EVEX.128.0F.W0 10 /r`
  - Instruction: `VMOVUPS xmm1 {k1}{z}`

**실행 방법**:
1. EVEX/VEX/REX로 시작하는 항목 처리
2. `/r`, `/rm` 등의 패턴 찾기
3. 그 다음 V로 시작하거나 연속된 문자가 명령어임을 인식

In [30]:
# 단계 2: Opcode와 Instruction 분리
def separate_opcode_and_instruction(inst_text):
    """
    Opcode와 Instruction을 분리합니다.
    
    예)  EVEX.128.0F.W0 10 /rVMOVUPS ... 
         → Opcode: EVEX.128.0F.W0 10 /r
    → Instruction: VMOVUPS ...
    """
    # EVEX/VEX/REX 접두사를 가진 경우
    match = re.match(r'^((?:EVEX|VEX|REX)[^V]*?/[rR])([A-Z].*?)(?:\s|$)', inst_text)
    
    if match:
        opcode = match.group(1).strip()
        rest_idx = match.end(1)
        instruction = inst_text[rest_idx:].strip()
        return opcode, instruction
    
    # Opcode 없음 (일반적인 경우)
    return None, inst_text

# 모든 명령어 처리
opcodes_list = []
instructions_separated = []

for inst in instructions:
    opcode, instruction = separate_opcode_and_instruction(inst)
    opcodes_list.append(opcode)
    instructions_separated.append(instruction)

# 통계
opcode_count = sum(1 for o in opcodes_list if o is not None)

print("=" * 80)
print("Opcode-Instruction 분리 결과")
print("=" * 80)
print(f"\n✓ Opcode 발견됨: {opcode_count}개")
print(f"✓ Opcode 없음 (일반 명령어): {len(instructions) - opcode_count}개")

# 분리 결과 샘플
print(f"\n📌 분리 결과 샘플 (Opcode가 있는 것들):")
count = 0
for orig, opc, inst in zip(instructions, opcodes_list, instructions_separated):
    if opc:
        print(f"\n  [{count+1}] 원본:")
        print(f"      {orig[:70]}")
        print(f"      Opcode: {opc[:50]}")
        print(f"      Instruction: {inst[:50]}")
        count += 1
        if count >= 3:
            break

Opcode-Instruction 분리 결과

✓ Opcode 발견됨: 2085개
✓ Opcode 없음 (일반 명령어): 3043개

📌 분리 결과 샘플 (Opcode가 있는 것들):

  [1] 원본:
      EVEX.128.0F.W0 10 /rVMOVUPS xmm1 {k1}{z}
      Opcode: EVEX.128.0F.W0 10 /r
      Instruction: VMOVUPS xmm1 {k1}{z}

  [2] 원본:
      EVEX.128.0F.W0 11 /rVMOVUPS xmm2/m128 {k1}{z}
      Opcode: EVEX.128.0F.W0 11 /r
      Instruction: VMOVUPS xmm2/m128 {k1}{z}

  [3] 원본:
      EVEX.128.0F.W0 12 /rVMOVHLPS xmm1
      Opcode: EVEX.128.0F.W0 12 /r
      Instruction: VMOVHLPS xmm1


## 섹션 3: Combined Operand 분리

**목표**: `/` 또는 `&`로 묶여있는 오퍼랜드를 분리

**패턴들**:
- `r/m16` → `r16`, `m16`
- `m16&16` → `m16` (유효성 검증)
- `xmm2/m128` → `xmm2`, `m128`
- `r32/m32` → `r32`, `m32`

In [31]:
# 단계 3: Combined Operand 분리 (명령어 확장)
def expand_combined_operands_with_metadata(instructions_list, opcodes_list, original_list):
    """
    r/m16 같은 combined operand를 가진 명령어를 여러 개의 명령어로 확장합니다.
    원본 정보도 함께 추적합니다.
    
    반환: (확장된_명령어, 확장된_오피코드, 확장된_원본)
    """
    expanded_instructions = []
    expanded_opcodes = []
    expanded_originals = []
    
    for inst, opcode, orig in zip(instructions_list, opcodes_list, original_list):
        variants = []
        
        # 1. r/m 패턴 분리 (r/m16, r/m32, r/m64 등)
        rm_pattern = re.search(r'\b(r/m(\d+))\b', inst)
        if rm_pattern:
            full_pattern = rm_pattern.group(1)  # "r/m16"
            size = rm_pattern.group(2)           # "16"
            var1 = inst.replace(full_pattern, f'r{size}')
            var2 = inst.replace(full_pattern, f'm{size}')
            variants = [var1, var2]
        
        # 2. 벡터/메모리 패턴 (xmm2/m128, ymm1/m256 등)
        elif re.search(r'[xyzr]\w+/m\d+', inst):
            vec_pattern = re.search(r'(\w+/m\d+)', inst)
            if vec_pattern:
                pattern = vec_pattern.group(1)
                parts = pattern.split('/')
                var1 = inst.replace(pattern, parts[0])
                var2 = inst.replace(pattern, parts[1])
                variants = [var1, var2]
        
        # 3. m&숫자 패턴 제거 (m16&16 → m16)
        elif re.search(r'\bm(\d+)&\d+\b', inst):
            cleaned = re.sub(r'\bm(\d+)&\d+\b', r'm\1', inst)
            variants = [cleaned]
        
        # variants가 없으면 원본 유지
        if not variants:
            variants = [inst]
        
        # 확장된 모든 variant에 대해 메타데이터 추가
        for variant in variants:
            expanded_instructions.append(variant)
            expanded_opcodes.append(opcode)
            expanded_originals.append(orig)
    
    return expanded_instructions, expanded_opcodes, expanded_originals

# 명령어 확장 실행
instructions_split, opcodes_split, originals_split = expand_combined_operands_with_metadata(
    instructions_separated, opcodes_list, instructions
)

print("=" * 80)
print("Combined Operand 분리 결과 (명령어 확장)")
print("=" * 80)

print(f"\n✓ 원본 명령어 수: {len(instructions)}")
print(f"✓ 분리 후 명령어 수: {len(instructions_split)}")
print(f"✓ 새로 생성된 명령어: {len(instructions_split) - len(instructions)}개")

# 비교 샘플
print(f"\n📌 분리 전후 비교 (하나의 명령어가 여러 줄로 분리):")

# instructions_separated에서 combined operand 찾기
samples = []
for i, inst in enumerate(instructions_separated):
    if 'r/m' in inst or '/m' in inst or '&' in inst:
        samples.append((i, inst))
        if len(samples) >= 5:
            break

for idx, original in samples:
    print(f"\n  【원본】{original}")
    
    # 분리 로직 재실행하여 결과 표시
    rm_pattern = re.search(r'\b(r/m(\d+))\b', original)
    if rm_pattern:
        full_pattern = rm_pattern.group(1)
        size = rm_pattern.group(2)
        var1 = original.replace(full_pattern, f'r{size}')
        var2 = original.replace(full_pattern, f'm{size}')
        print(f"    → {var1}")
        print(f"    → {var2}")
    else:
        vec_pattern = re.search(r'(\w+/m\d+)', original)
        if vec_pattern:
            pattern = vec_pattern.group(1)
            parts = pattern.split('/')
            var1 = original.replace(pattern, parts[0])
            var2 = original.replace(pattern, parts[1])
            print(f"    → {var1}")
            print(f"    → {var2}")
        else:
            amp_match = re.search(r'\bm(\d+)&\d+\b', original)
            if amp_match:
                cleaned = re.sub(r'\bm(\d+)&\d+\b', r'm\1', original)
                print(f"    → {cleaned}")

Combined Operand 분리 결과 (명령어 확장)

✓ 원본 명령어 수: 5128
✓ 분리 후 명령어 수: 6080
✓ 새로 생성된 명령어: 952개

📌 분리 전후 비교 (하나의 명령어가 여러 줄로 분리):

  【원본】ADC r/m16
    → ADC r16
    → ADC m16

  【원본】ADC r/m32
    → ADC r32
    → ADC m32

  【원본】ADC r/m64
    → ADC r64
    → ADC m64

  【원본】ADC r/m81
    → ADC r81
    → ADC m81

  【원본】ADD r/m16
    → ADD r16
    → ADD m16


## 섹션 4: 오퍼랜드 추출 및 중복 제거

**목표**: 모든 명령어에서 오퍼랜드를 추출한 후, Set을 사용하여 중복 제거

**추추출할 패턴**:
- 일반 레지스터: `r8`, `r16`, `r32`, `r64`, `al`, `ax`, `eax`, `rax`
- 벡터 레지스터: `xmm0-31`, `ymm0-31`, `zmm0-31`
- 메모리 클래스: `m8`, `m16`, `m32`, `m64`, `m128`, `m256`, `m512`
- 즉시값: `imm8`, `imm16`, `imm32`
- 기타: `rel8`, `rel16`, `rel32`, `ptr16:16` 등

In [32]:
# 단계 4: 오퍼랜드 추출 및 중복 제거
def extract_operands(text):
    """
    텍스트에서 오퍼랜드를 추출합니다.
    정규식을 사용하여 다양한 오퍼랜드 형식을 인식합니다.
    """
    operands = set()
    
    # 레지스터 (r8, r16, r32, r64, al, ax, eax, rax 등)
    reg_patterns = [
        r'\b(r[0-7]\b|\br\d+)',  # r8, r16 등
        r'\b([a-z]{2,3}h?)\b',   # al, ah, ax, eax, rax 등
        r'\b([xy]mm\d+)\b',       # xmm0-31, ymm0-31
        r'\b(zmm\d+)\b',          # zmm0-31
        r'\b(rip)\b',             # rip
    ]
    
    # 메모리 클래스 (m8, m16, m32, m64, m128 등)
    mem_patterns = [
        r'\b(m\d+)\b',            # m8, m16, m32, m64, m128 등
    ]
    
    # 즉시값 (imm8, imm16, imm32)
    imm_patterns = [
        r'\b(imm\d+)\b',          # imm8, imm16, imm32
    ]
    
    # 기타 (ptr, rel, vm 등)
    other_patterns = [
        r'\b(ptr\d+:\d+)\b',      # ptr16:16, ptr16:32 등
        r'\b(rel\d+)\b',          # rel8, rel16, rel32
        r'\b(vm\d+[a-z])\b',      # vm32x, vm64x 등
        r'\b(k[0-7])\b',          # k0-k7 (마스크 레지스터)
        r'\b(reg)\b',             # reg (일반 레지스터 참조)
    ]
    
    # 모든 패턴 적용
    for pattern in reg_patterns + mem_patterns + imm_patterns + other_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        operands.update([m.lower() for m in matches])
    
    return operands

# 모든 명령어에서 오퍼랜드 추출
all_operands = set()
for inst_text in instructions_split:
    operands = extract_operands(inst_text)
    all_operands.update(operands)

# 정렬
sorted_operands = sorted(list(all_operands))

print("=" * 80)
print("오퍼랜드 추출 및 중복 제거")
print("=" * 80)
print(f"\n✓ 추출된 총 오퍼랜드 개수: {len(all_operands)}")
print(f"\n📌 추출된 오퍼랜드 목록 (상위 30개):")
for i, op in enumerate(sorted_operands[:30], 1):
    print(f"  {i:2}. {op}")

if len(sorted_operands) > 30:
    print(f"\n  ... 외 {len(sorted_operands) - 30}개")

오퍼랜드 추출 및 중복 제거

✓ 추출된 총 오퍼랜드 개수: 229

📌 추출된 오퍼랜드 목록 (상위 30개):
   1. aaa
   2. aad
   3. aam
   4. aas
   5. adc
   6. add
   7. ae
   8. aes
   9. al
  10. and
  11. avx
  12. ax
  13. bbb
  14. bc
  15. bd
  16. bit
  17. bnd
  18. bsf
  19. bsr
  20. bt
  21. btc
  22. btr
  23. bts
  24. bx
  25. by
  26. ca
  27. cb
  28. cbw
  29. cc
  30. cd

  ... 외 199개


## 섹션 5: 오퍼랜드 의미별 분류

**목표**: 추출한 오퍼랜드를 이해하기 쉬운 범주로 분류

**분류 카테고리**:
1. **레지스터 8비트**: al, ah, bl, bh, ...
2. **레지스터 16비트**: r16, ax, bx, ...
3. **레지스터 32비트**: r32, eax, ebx, ...
4. **레지스터 64비트**: r64, rax, rbx, ...
5. **벡터 레지스터**: xmm, ymm, zmm
6. **메모리 8비트**: m8
7. **메모리 16비트**: m16
8. **메모리 32비트**: m32, m64
9. **메모리 128비트 이상**: m128, m256, m512
10. **즉시값**: imm8, imm16, imm32
11. **주소 지정**: ptr, rel, rip
12. **마스크및 기타**: k0-k7, reg 등

In [33]:
# 단계 5: 오퍼랜드 의미별 분류
def classify_operands(operands_set):
    """
    오퍼랜드를 의미에 따라 분류합니다.
    """
    classification = {
        'Register_8bit': set(),
        'Register_16bit': set(),
        'Register_32bit': set(),
        'Register_64bit': set(),
        'VectorRegister_XMM': set(),
        'VectorRegister_YMM': set(),
        'VectorRegister_ZMM': set(),
        'Memory_8bit': set(),
        'Memory_16bit': set(),
        'Memory_32bit': set(),
        'Memory_64bit': set(),
        'Memory_128bit_plus': set(),
        'Immediate': set(),
        'AddressingMode': set(),
        'MaskRegister': set(),
        'Other': set(),
    }
    
    for op in operands_set:
        # 8비트 레지스터
        if op in ['al', 'ah', 'bl', 'bh', 'cl', 'ch', 'dl', 'dh', 'sil', 'dil', 'spl', 'bpl', 'r8l', 'r9l', 'r10l', 'r11l', 'r12l', 'r13l', 'r14l', 'r15l'] or re.match(r'^r\d+\b', op):
            if op.endswith('l'):  # 8-bit suffix
                classification['Register_8bit'].add(op)
            elif re.match(r'^r8$', op):
                classification['Register_8bit'].add(op)
        
        # 16비트 레지스터
        elif op in ['r16', 'ax', 'bx', 'cx', 'dx', 'si', 'di', 'sp', 'bp'] or op == 'r16':
            classification['Register_16bit'].add(op)
        
        # 32비트 레지스터
        elif op in ['r32', 'eax', 'ebx', 'ecx', 'edx', 'esi', 'edi', 'esp', 'ebp'] or op == 'r32':
            classification['Register_32bit'].add(op)
        
        # 64비트 레지스터
        elif op in ['r64', 'rax', 'rbx', 'rcx', 'rdx', 'rsi', 'rdi', 'rsp', 'rbp'] or op == 'r64' or re.match(r'^r\d+$', op) and not op.endswith('l'):
            classification['Register_64bit'].add(op)
        
        # XMM 레지스터
        elif 'xmm' in op:
            classification['VectorRegister_XMM'].add(op)
        
        # YMM 레지스터
        elif 'ymm' in op:
            classification['VectorRegister_YMM'].add(op)
        
        # ZMM 레지스터
        elif 'zmm' in op:
            classification['VectorRegister_ZMM'].add(op)
        
        # 메모리
        elif op == 'm8':
            classification['Memory_8bit'].add(op)
        elif op == 'm16':
            classification['Memory_16bit'].add(op)
        elif op in ['m32', 'm64']:
            classification['Memory_32bit'].add(op)
        elif op in ['m128', 'm256', 'm512']:
            classification['Memory_128bit_plus'].add(op)
        
        # 즉시값
        elif 'imm' in op:
            classification['Immediate'].add(op)
        
        # 주소 지정
        elif 'ptr' in op or 'rel' in op or op == 'rip' or 'vm' in op:
            classification['AddressingMode'].add(op)
        
        # 마스크 레지스터
        elif 'k' in op and re.match(r'^k[0-7]$', op):
            classification['MaskRegister'].add(op)
        
        # 기타
        else:
            classification['Other'].add(op)
    
    return classification

classified = classify_operands(all_operands)

print("=" * 80)
print("오퍼랜드 분류 결과")
print("=" * 80)

for category, operands in classified.items():
    if operands:
        print(f"\n🔹 {category}: {len(operands)}개")
        # 상위 5개만 보여주기
        for op in sorted(list(operands))[:5]:
            print(f"   • {op}")
        if len(operands) > 5:
            print(f"   ... 외 {len(operands) - 5}개")

오퍼랜드 분류 결과

🔹 Register_8bit: 3개
   • al
   • cl
   • r8

🔹 Register_16bit: 3개
   • ax
   • bx
   • dx

🔹 Register_32bit: 2개
   • eax
   • edx

🔹 Register_64bit: 3개
   • rax
   • rbx
   • rdx

🔹 VectorRegister_XMM: 6개
   • xmm
   • xmm0
   • xmm1
   • xmm2
   • xmm3
   ... 외 1개

🔹 VectorRegister_YMM: 5개
   • ymm
   • ymm1
   • ymm2
   • ymm3
   • ymm4

🔹 VectorRegister_ZMM: 3개
   • zmm1
   • zmm2
   • zmm3

🔹 Memory_8bit: 1개
   • m8

🔹 Memory_16bit: 1개
   • m16

🔹 Memory_32bit: 2개
   • m32
   • m64

🔹 Memory_128bit_plus: 3개
   • m128
   • m256
   • m512

🔹 Immediate: 5개
   • imm16
   • imm32
   • imm64
   • imm8
   • imm81

🔹 AddressingMode: 11개
   • ptr16:16
   • ptr16:32
   • rel16
   • rel32
   • rel8
   ... 외 6개

🔹 MaskRegister: 3개
   • k1
   • k2
   • k3

🔹 Other: 170개
   • aaa
   • aad
   • aam
   • aas
   • adc
   ... 외 165개


## 섹션 6: 최종 리포트 및 결과 내보내기

**출력 파일**:
1. `step2_final_instruction_opcode_split.csv` - Opcode와 Instruction이 분리된 데이터
2. `step3_operands_raw.csv` - 추출되고 분류된 모든 오퍼랜드
3. `step3_operands_classified.csv` - 의미별로 분류된 오퍼랜드

**최종 통계**:
- 원본 명령어 수
- Opcode 분리 성공 건수
- 추출된 오퍼랜드 종류별 개수

In [34]:
# 단계 6: 최종 결과 내보내기

# 1. Opcode와 Instruction이 분리된 데이터 (확장된 버전)
result_df = pd.DataFrame({
    'Original': originals_split,
    'Opcode': opcodes_split,
    'Instruction': instructions_split
})

result_df.to_csv('step2_final_instruction_opcode_split.csv', index=False, encoding='utf-8')
print(f"✓ 저장됨: step2_final_instruction_opcode_split.csv ({len(result_df)}줄)")

# 2. 추출된 오퍼랜드 (분류 전)
raw_operands_df = pd.DataFrame({
    'Operand': sorted_operands
})
raw_operands_df.to_csv('step3_operands_raw.csv', index=False, encoding='utf-8')
print(f"✓ 저장됨: step3_operands_raw.csv ({len(sorted_operands)}개)")

# 3. 분류된 오퍼랜드
classified_operands = []
for category, operands in classified.items():
    for op in operands:
        classified_operands.append({'Category': category, 'Operand': op})

classified_df = pd.DataFrame(classified_operands)
classified_df = classified_df.sort_values(['Category', 'Operand']).reset_index(drop=True)
classified_df.to_csv('step3_operands_classified.csv', index=False, encoding='utf-8')
print(f"✓ 저장됨: step3_operands_classified.csv")

# 최종 통계
print("\n" + "=" * 80)
print("최종 통계 요약")
print("=" * 80)

print(f"\n📊 명령어 처리 통계:")
print(f"  • 원본 명령어 수: {len(instructions)}")
print(f"  • Opcode 분리 성공: {sum(1 for o in opcodes_list if o is not None)}개")
print(f"  • Opcode 미분리 (일반 명령어): {sum(1 for o in opcodes_list if o is None)}개")
print(f"  • Combined Operand 분리 후: {len(instructions_split)}개")
print(f"  • 추가 생성된 명령어: {len(instructions_split) - len(instructions)}개")

print(f"\n📊 오퍼랜드 추출 통계:")
print(f"  • 추출된 총 오퍼랜드: {len(all_operands)}")

print(f"\n📊 오퍼랜드 분류 통계:")
for category in sorted(classified.keys()):
    count = len(classified[category])
    if count > 0:
        print(f"  • {category:25}: {count:3}개")

print("\n" + "=" * 80)
print("✅ 모든 처리 완료!")
print("=" * 80)
print(f"\n📝 생성된 파일:")
print(f"  1. step2_final_instruction_opcode_split.csv - {len(result_df)}줄")
print(f"  2. step3_operands_raw.csv - {len(sorted_operands)}개")
print(f"  3. step3_operands_classified.csv - {len(classified_df)}개")

✓ 저장됨: step2_final_instruction_opcode_split.csv (6080줄)
✓ 저장됨: step3_operands_raw.csv (229개)
✓ 저장됨: step3_operands_classified.csv

최종 통계 요약

📊 명령어 처리 통계:
  • 원본 명령어 수: 5128
  • Opcode 분리 성공: 2085개
  • Opcode 미분리 (일반 명령어): 3043개
  • Combined Operand 분리 후: 6080개
  • 추가 생성된 명령어: 952개

📊 오퍼랜드 추출 통계:
  • 추출된 총 오퍼랜드: 229

📊 오퍼랜드 분류 통계:
  • AddressingMode           :  11개
  • Immediate                :   5개
  • MaskRegister             :   3개
  • Memory_128bit_plus       :   3개
  • Memory_16bit             :   1개
  • Memory_32bit             :   2개
  • Memory_8bit              :   1개
  • Other                    : 170개
  • Register_16bit           :   3개
  • Register_32bit           :   2개
  • Register_64bit           :   3개
  • Register_8bit            :   3개
  • VectorRegister_XMM       :   6개
  • VectorRegister_YMM       :   5개
  • VectorRegister_ZMM       :   3개

✅ 모든 처리 완료!

📝 생성된 파일:
  1. step2_final_instruction_opcode_split.csv - 6080줄
  2. step3_operands_raw.csv - 229개
  3. step3_oper